## Gradiente de Brown Almost Linear (ALMOST)

La función **ALMOST** (Brown Almost Linear) se define como

$$f(x) = \sum_{i=1}^{n-1}\Big(x_i + S - (n+1)\Big)^2 + \Big(P - 1\Big)^2,$$

donde
$$S = \sum_{j=1}^{n} x_j, \qquad P = \prod_{j=1}^{n} x_j.$$

Definimos los residuos
$$r_i = x_i + S - (n+1)\quad (i=1,\dots,n-1), \qquad r_n = P - 1,$$
y denotamos $R = \sum_{i=1}^{n-1} r_i$.

Aplicando la regla de la cadena, para $k=1,\dots,n-1$ se cumple
$$\frac{\partial r_i}{\partial x_k} = \delta_{ik} + 1 \quad (i=1,\dots,n-1), \qquad \frac{\partial r_n}{\partial x_k} = P_k := \prod_{j\neq k} x_j,$$
y por tanto
$$\frac{\partial f}{\partial x_k} = 2\big(R + r_k\big) + 2\,r_n\,P_k \quad (k=1,\dots,n-1),$$
$$\frac{\partial f}{\partial x_n} = 2R + 2\,r_n\,P_n.$$

**Observación.** El término $P_k = P/x_k$ (producto de todas las variables excepto $x_k$) introduce un acoplamiento global entre todas las coordenadas: un cambio en una variable afecta a los $n-1$ residuos lineales y al término del producto. Esto genera valles curvos y estrechos donde el antigradiente no apunta directamente al mínimo.

In [38]:
import numpy as np

def almost_f(x):
    """
    Evalúa la función Brown Almost Linear (ALMOST).
    x: array de longitud n ≥ 1.
    """
    n = len(x)
    S = np.sum(x)
    r = x[:n-1] + S - (n+1)        # primeros n-1 residuos
    prod = np.prod(x)
    return np.sum(r**2) + (prod - 1.0)**2


def almost_grad(x):
    """
    Gradiente analítico de ALMOST.
    Devuelve un array de la misma dimensión que x.
    """
    n = len(x)
    S = np.sum(x)
    r = np.zeros(n)
    r[:n-1] = x[:n-1] + S - (n+1)   # residuos lineales
    prod = np.prod(x)
    r[n-1] = prod - 1.0              # residuo del producto
    R = np.sum(r[:n-1])

    g = np.zeros(n)
    for k in range(n):
        # P_k = producto de todos los x_j con j != k (sin dividir por cero)
        if x[k] != 0:
            Pk = prod / x[k]
        else:
            Pk = np.prod(np.delete(x, k))

        if k < n-1:
            g[k] = 2.0 * (R + r[k]) + 2.0 * r[n-1] * Pk
        else:
            g[k] = 2.0 * R + 2.0 * r[n-1] * Pk
    return g

# ----- Verificación con n=4, x0=(0.5,0.5,0.5,0.5) -----
x0 = np.array([0.5, 0.5, 0.5, 0.5])
print(f"f(x0)       = {almost_f(x0):.8f}")            # esperado: 19.62890625
print(f"grad(x0)    = {almost_grad(x0)}")             # esperado: (-20.234375, -20.234375, -20.234375, -15.234375)
print(f"||grad(x0)||= {np.linalg.norm(almost_grad(x0)):.6f}")  # esperado: ≈ 38.2153

f(x0)       = 19.62890625
grad(x0)    = [-20.234375 -20.234375 -20.234375 -15.234375]
||grad(x0)||= 38.214866


## Solución práctica: Máximo Descenso con búsqueda lineal de Armijo

El algoritmo de máximo descenso con retroceso de Armijo se implementa exactamente igual que para SINGX (ver notebook de SINGX), ya que es genérico para cualquier par $(f,\nabla f)$.

**Resultados esperados para $n=4$, $x_0=(0.5,0.5,0.5,0.5)$:**
- $f(x_0) = 19.62890625$, $\|\nabla f(x_0)\| \approx 38.2153$.
- $d_0 \approx (0.5295,\,0.5295,\,0.5295,\,0.3987)$.
- La primera iteración **acepta $\alpha_0=1$** y produce $x_1\approx(1.0295,1.0295,1.0295,0.8986)$ con $f(x_1)\approx 1.2\times 10^{-3}$.
- A partir de la segunda iteración, $\alpha_k$ cae drásticamente (p. ej.\ $\alpha_1\approx 3.9\times 10^{-3}$) porque $d_1$ es casi ortogonal a la dirección que lleva al mínimo: **patrón de zigzag**.
- La sucesión converge a $x^*=(1,1,1,1)$ con $f(x^*)=0$, pero de forma lenta por el acoplamiento global de la función.

In [39]:
def max_descenso(f, grad, x0, tol=1e-6, max_iter=20000,
                 c1=1e-4, rho=0.5, verbose=True, print_every=10):
    """
    Método de Máximo Descenso con búsqueda lineal de Armijo (backtracking).

    Devuelve
    --------
    x_opt   : aproximación al mínimo
    history : diccionario con listas de iterados, valores de f, ||grad||, alpha
    """
    x = x0.copy().astype(float)
    history = {'x': [x.copy()],
               'f': [f(x)],
               'gnorm': [np.linalg.norm(grad(x))],
               'alpha': []}

    for k in range(max_iter):
        g = grad(x)
        gnorm = np.linalg.norm(g)

        if gnorm < tol:
            if verbose:
                print(f"Convergencia en {k} iteraciones, ||g|| = {gnorm:.2e}")
            break

        d = -g / gnorm                 # dirección de máximo descenso
        fk = f(x)
        gd = g @ d                     # = -||g|| < 0
        alpha = 1.0

        # Backtracking de Armijo
        while f(x + alpha*d) > fk + c1*alpha*gd:
            alpha *= rho

        x = x + alpha*d
        history['x'].append(x.copy())
        history['f'].append(f(x))
        history['gnorm'].append(np.linalg.norm(grad(x)))
        history['alpha'].append(alpha)

        if verbose and (k < 5 or k % print_every == 0):
            print(f"k={k:4d}  f={history['f'][-1]:.6e}  "
                  f"||g||={history['gnorm'][-1]:.3e}  alpha={alpha:.3e}")

    return x, history

In [40]:
# --------- Ejecución del Método de Máximo Descenso sobre ALMOST ---------
x0 = np.array([0.5, 0.5, 0.5, 0.5])

print(f"x0          = {x0}")
print(f"f(x0)       = {almost_f(x0):.8f}")   # 19.62890625
print(f"||grad(x0)||= {np.linalg.norm(almost_grad(x0)):.6f}")  # ≈ 38.2153
print()

x_opt, hist = max_descenso(almost_f, almost_grad, x0,
                           tol=1e-8, max_iter=20000,
                           c1=1e-4, rho=0.5,
                           verbose=True, print_every=10)

print(f"\n--- Resultado final ---")
print(f"x_opt     = {x_opt}")
print(f"f(x_opt)  = {almost_f(x_opt):.6e}")
print(f"||grad||  = {np.linalg.norm(almost_grad(x_opt)):.3e}")
print(f"Iteraciones totales: {len(hist['f'])-1}")
print(f"\nPrimeras iteraciones:")
for k in range(min(6, len(hist['f']))):
    print(f"  k={k:2d}  f={hist['f'][k]:.6e}  ||g||={hist['gnorm'][k]:.3e}")

x0          = [0.5 0.5 0.5 0.5]
f(x0)       = 19.62890625
||grad(x0)||= 38.214866

k=   0  f=1.207021e-03  ||g||=1.754e-01  alpha=1.000e+00
k=   1  f=8.647360e-04  ||g||=1.855e-02  alpha=3.906e-03
k=   2  f=6.681006e-04  ||g||=1.552e-01  alpha=3.125e-02
k=   3  f=4.078252e-04  ||g||=2.546e-02  alpha=3.906e-03
k=   4  f=4.003681e-04  ||g||=2.041e-02  alpha=9.766e-04
k=  10  f=3.690106e-04  ||g||=2.219e-02  alpha=9.766e-04
k=  20  f=3.227262e-04  ||g||=2.223e-02  alpha=9.766e-04
k=  30  f=2.834631e-04  ||g||=2.225e-02  alpha=9.766e-04
k=  40  f=2.499606e-04  ||g||=2.226e-02  alpha=9.766e-04
k=  50  f=2.212170e-04  ||g||=2.227e-02  alpha=9.766e-04
k=  60  f=1.964344e-04  ||g||=2.228e-02  alpha=9.766e-04
k=  70  f=1.749705e-04  ||g||=2.229e-02  alpha=9.766e-04
k=  80  f=1.563043e-04  ||g||=2.230e-02  alpha=9.766e-04
k=  90  f=1.400100e-04  ||g||=2.231e-02  alpha=9.766e-04
k= 100  f=1.257366e-04  ||g||=2.231e-02  alpha=9.766e-04
k= 110  f=1.131936e-04  ||g||=2.232e-02  alpha=9.766e-04
k= 12

# Método de Newton aplicado a Brown Almost Linear (ALMOST)

## Gradiente de ALMOST

La función **ALMOST** es
$$f(x)=\sum_{i=1}^{n-1}\big(x_i+S-(n+1)\big)^2+(P-1)^2,\qquad S=\sum_{j=1}^n x_j,\quad P=\prod_{j=1}^n x_j.$$
Definimos los residuos $r_i=x_i+S-(n+1)$ para $i=1,\dots,n-1$, $r_n=P-1$, y denotamos $R=\sum_{i=1}^{n-1}r_i$. Entonces
$$\frac{\partial f}{\partial x_k}=2(R+r_k)+2r_n P_k\quad(k=1,\dots,n-1),\qquad\frac{\partial f}{\partial x_n}=2R+2r_n P_n,$$
donde $P_k=\prod_{j\neq k}x_j$.

## Hessiana de ALMOST

Aplicando la regla de la cadena y teniendo en cuenta que $\partial r_i/\partial x_k=\delta_{ik}+1$ y $\partial^2 r_n/\partial x_k\partial x_l=0$ si $k=l$, $=P_{kl}:=\prod_{j\notin\{k,l\}}x_j$ si $k\neq l$, se obtiene:
$$H_{kl}=2\sum_{i=1}^{n-1}(\delta_{ik}+1)(\delta_{il}+1)+2P_kP_l+2r_n\cdot\begin{cases}P_{kl}, & k\neq l\\ 0, & k=l\end{cases}$$
Evaluando la suma:
- Si $k,l\le n-1$: $H_{kl}=2(n+1+\delta_{kl})+2P_kP_l+2r_nP_{kl}\cdot\mathbb{1}_{k\neq l}$.
- Si $k\le n-1$, $l=n$: $H_{kl}=2n+2P_kP_n+2r_nP_{kl}$.
- $H_{nn}=2(n-1)+2P_n^2$.

## Ejemplo de una iteración ($n=4$, $x_0=(0.5,0.5,0.5,0.5)^t$)

$S=2$, $r_i=-2.5$ $(i=1,2,3)$, $R=-7.5$, $P=0.0625$, $r_4=-0.9375$, $P_k=0.125$ para todo $k$.
$$f(x_0)=3\cdot(-2.5)^2+(-0.9375)^2\approx 19.6289,$$
$$\nabla f(x_0)=(-20.2344,\,-20.2344,\,-20.2344,\,-15.2344)^t,\quad\|\nabla f(x_0)\|\approx 38.2153,$$
$$\nabla^2 f(x_0)=\begin{pmatrix}12.031&9.5625&9.5625&7.5625\\9.5625&12.031&9.5625&7.5625\\9.5625&9.5625&12.031&7.5625\\7.5625&7.5625&7.5625&6.031\end{pmatrix},$$
que es definida positiva. Se resuelve $\nabla^2 f(x_0)d_0=-\nabla f(x_0)$, se actualiza $x_1=x_0+d_0$ y se observa una reducción drástica de \(f\) (de \(19.63\) a valores del orden de \(10^{-2}\)). La convergencia es cuadrática en una vecindad de \(x^*=(1,1,1,1)\), donde \(\nabla^2 f(x^*)\) es definida positiva (autovalores \(2,\,2,\,0.0871,\,45.913\) para \(n=4\)).

In [41]:
def almost_f(x):
    """Función ALMOST."""
    n = len(x); S = np.sum(x)
    r = x[:n-1] + S - (n+1)
    return np.sum(r**2) + (np.prod(x) - 1.0)**2

def almost_grad(x):
    """Gradiente analítico de ALMOST."""
    n = len(x); S = np.sum(x); P = np.prod(x)
    r = np.zeros(n)
    r[:n-1] = x[:n-1] + S - (n+1)
    r[n-1] = P - 1.0
    R = np.sum(r[:n-1])
    Pk = np.array([P/x[k] if x[k] != 0 else np.prod(np.delete(x, k)) for k in range(n)])
    g = np.zeros(n)
    for k in range(n-1):
        g[k] = 2*(R + r[k]) + 2*r[n-1]*Pk[k]
    g[n-1] = 2*R + 2*r[n-1]*Pk[n-1]
    return g

def almost_hess(x):
    """Hessiana analítica de ALMOST."""
    n = len(x); S = np.sum(x); P = np.prod(x)
    r = np.zeros(n)
    r[:n-1] = x[:n-1] + S - (n+1)
    r[n-1] = P - 1.0
    rn = r[n-1]
    Pk = np.array([P/x[k] if x[k] != 0 else np.prod(np.delete(x, k)) for k in range(n)])
    H = np.zeros((n, n))
    for k in range(n-1):
        for l in range(n-1):
            H[k, l] = 2*(n + 1 + (1 if k == l else 0)) + 2*Pk[k]*Pk[l]
            if k != l:
                H[k, l] += 2*rn*np.prod(np.delete(x, [k, l]))
        H[k, n-1] = 2*n + 2*Pk[k]*Pk[n-1] + 2*rn*np.prod(np.delete(x, [k, n-1]))
        H[n-1, k] = H[k, n-1]
    H[n-1, n-1] = 2*(n-1) + 2*Pk[n-1]**2
    return H

In [42]:
# ----- Verificación con n=4, x0=(0.5,0.5,0.5,0.5) -----
x0 = np.array([0.5, 0.5, 0.5, 0.5])
print(f"f(x0)       = {almost_f(x0):.8f}")            # esperado: 19.62890625
print(f"grad(x0)    = {almost_grad(x0)}")             # esperado: (-20.234375,)*3 + (-15.234375,)
print(f"||grad(x0)||= {np.linalg.norm(almost_grad(x0)):.6f}")  # esperado: ≈ 38.2153
print(f"H(x0)       =\n{np.round(almost_hess(x0), 6)}")
print(f"autovalores = {np.linalg.eigvalsh(almost_hess(x0))}")  # todos > 0
print()

# Newton desde x0
x_opt, hist = newton_method(almost_f, almost_grad, almost_hess, x0,
                            tol=1e-12, max_iter=50, verbose=True)

print(f"\n--- Resultado final ---")
print(f"x_opt      = {x_opt}")                # esperado: → (1,1,1,1)
print(f"f(x_opt)   = {almost_f(x_opt):.2e}")  # esperado: → 0
print(f"||grad||   = {np.linalg.norm(almost_grad(x_opt)):.2e}")
print(f"Iteraciones: {len(hist['f'])-1}")

# Hessiana en el óptimo (debe ser definida positiva)
print(f"\nAutovalores H(1,1,1,1) = {np.linalg.eigvalsh(almost_hess(np.ones(4)))}")
# esperado: ≈ (0.0871, 2, 2, 45.913)  -> todos > 0

f(x0)       = 19.62890625
grad(x0)    = [-20.234375 -20.234375 -20.234375 -15.234375]
||grad(x0)||= 38.214866
H(x0)       =
[[12.03125  9.5625   9.5625   7.5625 ]
 [ 9.5625  12.03125  9.5625   7.5625 ]
 [ 9.5625   9.5625  12.03125  7.5625 ]
 [ 7.5625   7.5625   7.5625   6.03125]]
autovalores = [ 0.44462812  2.46875     2.46875    36.74287188]

k=0: f=6.276738e-02, ||g||=2.457e+00, ||d||=1.197e+00
k=1: f=3.337418e-04, ||g||=5.393e-02, ||d||=5.653e-02
k=2: f=8.077751e+00, ||g||=2.650e+01, ||d||=6.776e+00
k=3: f=3.273639e+00, ||g||=1.006e+01, ||d||=7.180e-01
k=4: f=1.692277e+00, ||g||=3.605e+00, ||d||=5.691e-01
k=5: f=1.174033e+00, ||g||=1.235e+00, ||d||=5.250e-01
k=6: f=1.032566e+00, ||g||=3.763e-01, ||d||=4.317e-01
k=7: f=1.004897e+00, ||g||=1.046e-01, ||d||=2.860e-01
k=8: f=1.000669e+00, ||g||=2.791e-02, ||d||=1.618e-01
k=9: f=1.000088e+00, ||g||=7.256e-03, ||d||=8.578e-02
k=10: f=1.000011e+00, ||g||=1.854e-03, ||d||=4.420e-02
k=11: f=1.000001e+00, ||g||=4.690e-04, ||d||=2.245e-02
k=12

# Método Cuasi-Newton (BFGS) aplicado a ALMOST

## Gradiente de ALMOST (recordatorio)

$$f(x)=\sum_{i=1}^{n-1}\big(x_i+S-(n+1)\big)^2+(P-1)^2,\quad S=\sum_j x_j,\;P=\prod_j x_j.$$
Con $r_i=x_i+S-(n+1)$ ($i\le n-1$), $r_n=P-1$, $R=\sum_{i\le n-1}r_i$, $P_k=\prod_{j\neq k}x_j$:
$$\frac{\partial f}{\partial x_k}=2(R+r_k)+2r_n P_k\;(k<n),\qquad\frac{\partial f}{\partial x_n}=2R+2r_n P_n.$$

## Hessiana de ALMOST (recordatorio)

$$H_{kl}=2\sum_{i=1}^{n-1}(\delta_{ik}+1)(\delta_{il}+1)+2P_kP_l+2r_n\cdot\begin{cases}P_{kl},&k\neq l\\0,&k=l\end{cases}$$
En el óptimo \(x^*=(1,\dots,1)\), \(H\) es definida positiva (para \(n=4\): autovalores \(\approx 0.0871,\,2,\,2,\,45.913\)).

## Ejemplo de una iteración ($n=4$, $x_0=(0.5,0.5,0.5,0.5)^t$)

- $S=2$, $r_i=-2.5$ $(i=1,2,3)$, $R=-7.5$, $P=0.0625$, $r_4=-0.9375$, $P_k=0.125$.
- $f(x_0)\approx 19.6289$, $\nabla f(x_0)\approx(-20.2344,-20.2344,-20.2344,-15.2344)^t$, $\|\nabla f(x_0)\|\approx 38.2153$.
- $\nabla^2 f(x_0)$ es definida positiva (todos sus autovalores $>0$).
- $d_0=-B_0 g_0=-g_0=(20.2344,20.2344,20.2344,15.2344)^t$.
- La búsqueda lineal de Wolfe acepta $\alpha_0=1$ en la primera iteración (la dirección apunta hacia $(1,1,1,1)$).
- Se calculan $s_0$, $y_0$ y se actualiza $B_1$ mediante BFGS.

## Condiciones de aplicabilidad

1. $f\in C^1(\mathbb{R}^n)$: se cumple ($f$ es polinomio, $C^\infty$).
2. $B_0$ simétrica y definida positiva: $B_0=I$ cumple.
3. Búsqueda lineal que garantice $y_k^t s_k>0$: se satisface con Wolfe ($0<c_1<c_2<1$).
4. Dirección \(d_k=-B_k g_k\) de descenso: $g_k^t d_k=-g_k^t B_k g_k<0$ porque $B_k$ es definida positiva.